# P7513 clustering Squidpy interactive plot

This notebook recreates a publication-style version of the `clustering_squidpy` spatial scatter and UMAP outputs for P7513 MERSCOPE.

It loads the pregenerated clustered AnnData object, then plots the spatial leiden scatter on the left and the leiden-only UMAP on the right. Edit `CLUSTER_STYLE` below to rename leiden clusters and assign your own colors.

## Environment

Run this notebook from the MerXen conda environment so `anndata`, `matplotlib`, and the local `merxen` package are available.

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy import sparse

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.plotting import save_figure


In [ ]:
PAIR_ID = "P7513"
PLATFORM = "MERSCOPE"
PLATFORM_DIR = PLATFORM.lower()

CLUSTERED_H5AD_PATH = (
    REPO_ROOT
    / "results"
    / PAIR_ID
    / "clustering_squidpy"
    / "clustering_squidpy_out"
    / PLATFORM_DIR
    / f"{PAIR_ID}_{PLATFORM}_clustered.h5ad"
)
OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "clustering_squidpy" / "clustering_squidpy_out" / PLATFORM_DIR
OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_spatial_umap_leiden_interactive.png"
OUTPUT_PDF_PATH = OUTPUT_PATH.with_suffix(".pdf")

print(f"Clustered AnnData: {CLUSTERED_H5AD_PATH}")
print(f"PNG output:        {OUTPUT_PATH}")
print(f"PDF output:        {OUTPUT_PDF_PATH}")


In [ ]:
adata = ad.read_h5ad(CLUSTERED_H5AD_PATH)

print(adata)
print(f"spatial coordinates: {'spatial' in adata.obsm}")
print(f"UMAP coordinates:    {'X_umap' in adata.obsm}")
print(f"leiden clusters:     {adata.obs['leiden'].nunique() if 'leiden' in adata.obs else 'missing'}")


In [ ]:
def _category_sort_key(value: object) -> tuple[int, float | str]:
    text = str(value)
    try:
        return (0, float(text))
    except ValueError:
        return (1, text)


def _cluster_categories(adata_obj: ad.AnnData, cluster_key: str = "leiden") -> list[str]:
    if cluster_key not in adata_obj.obs:
        raise KeyError(f"Expected adata.obs[{cluster_key!r}] for cluster labels.")
    labels = adata_obj.obs[cluster_key]
    if isinstance(labels.dtype, pd.CategoricalDtype):
        categories = [str(category) for category in labels.cat.categories]
    else:
        categories = [str(category) for category in pd.unique(labels.astype(str))]
    return sorted(categories, key=_category_sort_key)


def _default_cluster_colors(adata_obj: ad.AnnData, categories: list[str], cluster_key: str = "leiden") -> dict[str, str]:
    stored_colors = list(adata_obj.uns.get(f"{cluster_key}_colors", []))
    if len(stored_colors) >= len(categories):
        return {category: stored_colors[index] for index, category in enumerate(categories)}

    cmap = plt.get_cmap("tab20")
    return {
        category: cmap(index % cmap.N)
        for index, category in enumerate(categories)
    }


def build_default_cluster_style(
    adata_obj: ad.AnnData,
    cluster_key: str = "leiden",
) -> dict[str, dict[str, str]]:
    """Create an editable style dict from the existing leiden categories and colors."""
    categories = _cluster_categories(adata_obj, cluster_key=cluster_key)
    colors = _default_cluster_colors(adata_obj, categories, cluster_key=cluster_key)
    return {
        category: {"name": f"Cluster {category}", "color": colors[category]}
        for category in categories
    }


CLUSTER_STYLE = build_default_cluster_style(adata, cluster_key="leiden")
CLUSTER_STYLE


## Edit Cluster Names And Colors

Edit the dictionary below to map leiden IDs to the cell type names and colors you want in the figure. Any clusters you leave out will fall back to `Cluster <id>` and the stored Scanpy color.

To merge multiple leiden clusters visually, give them the same `name` and `color`. They will be plotted with one shared color and collapsed to one legend entry.

In [ ]:
# Replace the names below with your preferred annotations.
# Colors use Matplotlib CSS4 color names, e.g. from matplotlib.colors.CSS4_COLORS.
#
# To merge clusters visually, assign the same name and color to multiple leiden IDs.
# The legend will collapse repeated names into one entry.
CLUSTER_STYLE.update({
    "0": {"name": "Fibroblasts", "color": "sienna"},
    "1": {"name": "Vascular cells", "color": "teal"},
    "2": {"name": "Microglia", "color": "goldenrod"},
    "3": {"name": "Vascular cells", "color": "teal"},
    "4": {"name": "Astrocytes", "color": "forestgreen"},
    "5": {"name": "Excitatory neurons", "color": "darkorange"},
    "6": {"name": "Inhibitory neurons", "color": "crimson"},
    "7": {"name": "Excitatory neurons", "color": "darkorange"},
    "8": {"name": "Oligodendrocytes", "color": "royalblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "plum"},
    "10": {"name": "Astrocytes", "color": "forestgreen"},
})
CLUSTER_STYLE


In [ ]:
def _normalize_cluster_style(
    adata_obj: ad.AnnData,
    cluster_style: dict[str, object] | None,
    cluster_key: str,
) -> dict[str, dict[str, str]]:
    categories = _cluster_categories(adata_obj, cluster_key=cluster_key)
    defaults = build_default_cluster_style(adata_obj, cluster_key=cluster_key)
    if cluster_style is None:
        return defaults

    normalized: dict[str, dict[str, str]] = {}
    for category in categories:
        style_value = cluster_style.get(category, defaults[category])
        if isinstance(style_value, str):
            normalized[category] = {"name": defaults[category]["name"], "color": style_value}
        elif isinstance(style_value, tuple | list) and len(style_value) == 2:
            normalized[category] = {"name": str(style_value[0]), "color": style_value[1]}
        elif isinstance(style_value, dict):
            normalized[category] = {
                "name": str(style_value.get("name", defaults[category]["name"])),
                "color": style_value.get("color", defaults[category]["color"]),
            }
        else:
            raise TypeError(
                "Cluster style values must be colors, (name, color) pairs, or "
                f"dicts with name/color keys. Got {type(style_value)!r} for cluster {category!r}."
            )
    return normalized


def _plot_cluster_points(
    ax: plt.Axes,
    coords: np.ndarray,
    labels: pd.Series,
    cluster_style: dict[str, dict[str, str]],
    *,
    point_size: float,
    alpha: float,
    add_legend: bool,
    legend_marker_size: float = 9.0,
    rasterized: bool = True,
) -> list[Line2D]:
    legend_handles: list[Line2D] = []
    legend_names_seen: set[str] = set()
    label_text = labels.astype(str)

    for cluster_id, style in cluster_style.items():
        mask = label_text.to_numpy() == cluster_id
        if not np.any(mask):
            continue
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            alpha=alpha,
            c=[style["color"]],
            edgecolors="none",
            linewidths=0,
            rasterized=rasterized,
        )
        if add_legend and style["name"] not in legend_names_seen:
            legend_names_seen.add(style["name"])
            legend_handles.append(
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    linestyle="",
                    label=style["name"],
                    markerfacecolor=style["color"],
                    markeredgecolor="none",
                    markersize=legend_marker_size,
                    alpha=1.0,
                )
            )
    return legend_handles


def plot_spatial_umap_leiden_interactive(
    adata_obj: ad.AnnData,
    output_path: Path | str | None = None,
    *,
    cluster_style: dict[str, object] | None = None,
    cluster_key: str = "leiden",
    spatial_point_size: float = 0.45,
    spatial_alpha: float = 0.65,
    umap_point_size: float = 0.45,
    umap_alpha: float = 0.65,
    legend_marker_size: float = 9.0,
    legend_fontsize: float = 10.0,
    legend_loc: str = "center left",
    legend_bbox_to_anchor: tuple[float, float] | None = (1.02, 0.5),
    figsize: tuple[float, float] = (12.0, 5.8),
    wspace: float = 0.05,
    dpi: int = 220,
) -> plt.Figure:
    """Plot spatial leiden labels and leiden-only UMAP with shared cluster colors."""
    if "spatial" not in adata_obj.obsm:
        raise KeyError("Expected adata.obsm['spatial'] for the spatial panel.")
    if "X_umap" not in adata_obj.obsm:
        raise KeyError("Expected adata.obsm['X_umap'] for the UMAP panel.")

    style = _normalize_cluster_style(adata_obj, cluster_style, cluster_key)
    labels = adata_obj.obs[cluster_key]
    spatial_xy = np.asarray(adata_obj.obsm["spatial"], dtype=float)
    umap_xy = np.asarray(adata_obj.obsm["X_umap"], dtype=float)

    fig, (spatial_ax, umap_ax) = plt.subplots(1, 2, figsize=figsize)

    _plot_cluster_points(
        spatial_ax,
        spatial_xy,
        labels,
        style,
        point_size=spatial_point_size,
        alpha=spatial_alpha,
        add_legend=False,
    )
    spatial_ax.set_aspect("equal", adjustable="box")
    spatial_ax.invert_yaxis()
    spatial_ax.set_title("")
    spatial_ax.set_xlabel("")
    spatial_ax.set_ylabel("")
    spatial_ax.set_xticks([])
    spatial_ax.set_yticks([])
    spatial_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in spatial_ax.spines.values():
        spine.set_visible(False)
    spatial_ax.set_frame_on(False)

    legend_handles = _plot_cluster_points(
        umap_ax,
        umap_xy,
        labels,
        style,
        point_size=umap_point_size,
        alpha=umap_alpha,
        add_legend=True,
        legend_marker_size=legend_marker_size,
    )
    umap_ax.set_title("")
    umap_ax.set_xlabel("UMAP1")
    umap_ax.set_ylabel("UMAP2")
    umap_ax.set_xticks([])
    umap_ax.set_yticks([])
    umap_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in umap_ax.spines.values():
        spine.set_visible(False)
    umap_ax.set_frame_on(False)

    if legend_handles:
        legend_kwargs = {
            "handles": legend_handles,
            "loc": legend_loc,
            "frameon": False,
            "fontsize": legend_fontsize,
        }
        if legend_bbox_to_anchor is not None:
            legend_kwargs["bbox_to_anchor"] = legend_bbox_to_anchor
        umap_ax.legend(**legend_kwargs)

    fig.subplots_adjust(wspace=wspace)
    if output_path is not None:
        save_figure(fig, output_path, dpi=dpi, bbox_inches="tight")
    return fig


In [ ]:
# Optional: save the interactive output.
# save_figure() is the repo helper used by production plots; it writes PNG plus a sibling PDF.

CLUSTER_STYLE = {
    "0": {"name": "Fibroblasts", "color": "sienna"},
    "1": {"name": "Vascular cells", "color": "teal"},
    "2": {"name": "Microglia", "color": "goldenrod"},
    "3": {"name": "Vascular cells", "color": "teal"},
    "4": {"name": "Astrocytes", "color": "forestgreen"},
    "5": {"name": "Excitatory neurons", "color": "darkorange"},
    "6": {"name": "Inhibitory neurons", "color": "crimson"},
    "7": {"name": "Excitatory neurons", "color": "darkorange"},
    "8": {"name": "Oligodendrocytes", "color": "royalblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "plum"},
    "10": {"name": "Astrocytes", "color": "forestgreen"},
}

CLUSTER_STYLE = {
    "0": {"name": "Fibroblasts", "color": "red"},
    "1": {"name": "Vascular cells", "color": "purple"},
    "2": {"name": "Microglia", "color": "gold"},
    "3": {"name": "Vascular cells", "color": "darkcyan"},
    "4": {"name": "Astrocytes", "color": "limegreen"},
    "5": {"name": "Excitatory neurons", "color": "orangered"},
    "6": {"name": "Inhibitory neurons", "color": "darkcyan"},
    "7": {"name": "Excitatory neurons", "color": "orangered"},
    "8": {"name": "Oligodendrocytes", "color": "dodgerblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "hotpink"},
    "10": {"name": "Astrocytes", "color": "limegreen"},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_spatial_umap_leiden_interactive(
    adata,
    output_path=OUTPUT_PATH,
    cluster_style=CLUSTER_STYLE,
    spatial_point_size=0.45,
    spatial_alpha=0.65,
    umap_point_size=0.45,
    umap_alpha=0.65,
    legend_marker_size=9.0,
    legend_fontsize=10,
    figsize=(12.0, 5.8),
    wspace=0.05,
)
print(f"Saved PNG: {OUTPUT_PATH}")
print(f"Saved PDF: {OUTPUT_PDF_PATH}")


## Marker Gene UMAP Grid

The marker defaults below were chosen from genes that are present in the shared Xenium/MERSCOPE panel and show cell-type-enriched expression in this dataset. The plotting function draws a five-column grid with one row per expected cell type and a per-gene expression color scale. The final section also builds a clustered dotplot across the shared panel so you can scan all genes for candidate markers.


In [ ]:
from scipy import sparse
from scipy.cluster.hierarchy import leaves_list, linkage

MARKER_GENES = {
    "Fibroblasts": ["DCN", "CEMIP", "FBLN1", "THBS1", "LAMA2"],
    "Vascular cells": ["FLT1", "PECAM1", "ABCC9", "KLF2", "CAV1"],
    "Microglia": ["P2RY12", "CX3CR1", "AIF1", "TREM2", "ITGAM"],
    "Astrocytes": ["AQP4", "SOX9", "GJA1", "FGFR3", "TTYH1"],
    "Excitatory neurons": ["SLC17A7", "CUX2", "RORB", "C1QL3", "TSHZ2"],
    "Inhibitory neurons": ["GAD1", "GAD2", "PVALB", "LHX6", "TAC1"],
    "Oligodendrocytes": ["MOBP", "MOG", "OPALIN", "MAG", "ERMN"],
    "Oligodendrocyte precursors": ["PDGFRA", "CSPG4", "VCAN", "OLIG2", "PTPRZ1"],
}
ANNOTATED_CELL_TYPE_ORDER = [
    "Oligodendrocytes",
    "Oligodendrocyte precursors",
    "Astrocytes",
    "Excitatory neurons",
    "Inhibitory neurons",
    "Microglia",
    "Fibroblasts",
    "Vascular cells",
]
ANNOTATED_CELL_TYPE_KEY = "annotated_cell_type"
SHARED_GENE_CORRESPONDENCE_PATH = (
    REPO_ROOT
    / "results"
    / PAIR_ID
    / "panel_gene_correspondence"
    / f"{PAIR_ID}_merscope_xenium_noncontrol_gene_correspondence.csv"
)


def _load_shared_genes(path: Path | str, gene_column: str = "gene") -> list[str]:
    shared_gene_table = pd.read_csv(path)
    if gene_column not in shared_gene_table:
        raise KeyError(f"Expected {gene_column!r} column in {path}.")
    return list(dict.fromkeys(shared_gene_table[gene_column].dropna().astype(str)))


def _resolve_gene_name(adata_obj: ad.AnnData, gene: str) -> str:
    """Resolve a marker gene against var_names and common gene annotation columns."""
    query = gene.upper()
    var_name_lookup = {str(var_name).upper(): str(var_name) for var_name in adata_obj.var_names}
    if query in var_name_lookup:
        return var_name_lookup[query]

    for column in ("gene", "gene_symbol", "gene_symbols", "symbol"):
        if column not in adata_obj.var:
            continue
        values = adata_obj.var[column].astype(str)
        matches = values.str.upper() == query
        if matches.any():
            return str(adata_obj.var_names[matches.to_numpy()][0])

    raise KeyError(f"Gene {gene!r} was not found in adata.var_names or gene annotation columns.")


def _shared_genes_in_adata(adata_obj: ad.AnnData, shared_genes: list[str]) -> tuple[list[str], list[str]]:
    resolved_genes: list[str] = []
    missing_genes: list[str] = []
    seen_resolved: set[str] = set()
    for gene in shared_genes:
        try:
            resolved_gene = _resolve_gene_name(adata_obj, gene)
        except KeyError:
            missing_genes.append(gene)
            continue
        if resolved_gene not in seen_resolved:
            resolved_genes.append(resolved_gene)
            seen_resolved.add(resolved_gene)
    return resolved_genes, missing_genes


def _validate_marker_genes(
    marker_genes: dict[str, list[str]],
    shared_genes: list[str],
    adata_obj: ad.AnnData,
) -> None:
    shared_gene_lookup = {gene.upper() for gene in shared_genes}
    missing_from_shared: list[str] = []
    missing_from_adata: list[str] = []

    for genes in marker_genes.values():
        for gene in genes:
            if gene.upper() not in shared_gene_lookup:
                missing_from_shared.append(gene)
            try:
                _resolve_gene_name(adata_obj, gene)
            except KeyError:
                missing_from_adata.append(gene)

    if missing_from_shared or missing_from_adata:
        messages: list[str] = []
        if missing_from_shared:
            messages.append("not in the shared Xenium/MERSCOPE panel: " + ", ".join(sorted(set(missing_from_shared))))
        if missing_from_adata:
            messages.append("not in AnnData: " + ", ".join(sorted(set(missing_from_adata))))
        raise ValueError("Marker gene validation failed; " + "; ".join(messages))


def add_annotated_cell_type(
    adata_obj: ad.AnnData,
    cluster_style: dict[str, object] | None = None,
    *,
    cluster_key: str = "leiden",
    cell_type_key: str = ANNOTATED_CELL_TYPE_KEY,
    cell_type_order: list[str] | None = None,
) -> str:
    """Add the editable cluster annotations as an ordered obs column."""
    if cluster_style is None:
        cluster_style = CLUSTER_STYLE
    style = _normalize_cluster_style(adata_obj, cluster_style, cluster_key)
    labels = adata_obj.obs[cluster_key].astype(str)
    cluster_to_cell_type = {cluster_id: entry["name"] for cluster_id, entry in style.items()}
    annotated = labels.map(cluster_to_cell_type)
    if annotated.isna().any():
        missing_clusters = sorted(set(labels[annotated.isna()]))
        raise ValueError(f"Missing cell-type annotations for clusters: {missing_clusters}")
    if cell_type_order is None:
        cell_type_order = list(dict.fromkeys(cluster_to_cell_type[cluster_id] for cluster_id in _cluster_categories(adata_obj, cluster_key)))
    adata_obj.obs[cell_type_key] = pd.Categorical(annotated, categories=cell_type_order, ordered=True)
    return cell_type_key


def _gene_expression_vector(adata_obj: ad.AnnData, gene: str, *, layer: str | None = None) -> np.ndarray:
    """Return a dense expression vector for one gene."""
    resolved_gene = _resolve_gene_name(adata_obj, gene)
    gene_index = int(adata_obj.var_names.get_loc(resolved_gene))
    matrix = adata_obj.layers[layer] if layer is not None else adata_obj.X
    values = matrix[:, gene_index]
    if sparse.issparse(values):
        values = values.toarray()
    return np.asarray(values).reshape(-1)


def plot_marker_expression_umap_grid(
    adata_obj: ad.AnnData,
    marker_genes: dict[str, list[str]] = MARKER_GENES,
    output_path: Path | str | None = None,
    *,
    layer: str | None = None,
    point_size: float = 0.35,
    alpha: float = 0.85,
    cmap: str = "viridis",
    expression_percentile: float = 99.5,
    figsize: tuple[float, float] = (18.0, 18.0),
    gene_title_fontsize: float = 10.0,
    cell_type_fontsize: float = 12.0,
    axis_label_fontsize: float = 9.0,
    colorbar_label_fontsize: float = 8.0,
    colorbar_tick_fontsize: float = 7.0,
    dpi: int = 220,
) -> plt.Figure:
    """Plot a UMAP marker-expression grid with one row per annotated cell type."""
    if "X_umap" not in adata_obj.obsm:
        raise KeyError("Expected adata.obsm['X_umap'] for marker UMAP plots.")

    column_counts = {len(genes) for genes in marker_genes.values()}
    if len(column_counts) != 1:
        raise ValueError(f"Expected each marker row to have the same number of genes; got {sorted(column_counts)}.")

    n_cols = column_counts.pop()
    n_rows = len(marker_genes)
    umap_xy = np.asarray(adata_obj.obsm["X_umap"], dtype=float)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, squeeze=False)

    for row_index, (cell_type, genes) in enumerate(marker_genes.items()):
        for col_index, gene in enumerate(genes):
            ax = axes[row_index, col_index]
            expression = _gene_expression_vector(adata_obj, gene, layer=layer)
            finite_expression = expression[np.isfinite(expression)]
            vmax = 1.0
            if finite_expression.size:
                vmax = float(np.percentile(finite_expression, expression_percentile))
                if vmax <= 0:
                    vmax = float(finite_expression.max()) if finite_expression.max() > 0 else 1.0

            scatter = ax.scatter(
                umap_xy[:, 0],
                umap_xy[:, 1],
                c=expression,
                s=point_size,
                alpha=alpha,
                cmap=cmap,
                vmin=0,
                vmax=vmax,
                edgecolors="none",
                linewidths=0,
                rasterized=True,
            )
            ax.set_title(gene, fontsize=gene_title_fontsize)
            ax.set_xlabel("UMAP1", fontsize=axis_label_fontsize)
            ax.set_ylabel("UMAP2", fontsize=axis_label_fontsize)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_frame_on(False)

            colorbar = fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.02)
            colorbar.set_label("Expression", fontsize=colorbar_label_fontsize)
            colorbar.ax.tick_params(labelsize=colorbar_tick_fontsize)

        axes[row_index, 0].text(
            -0.18,
            0.5,
            cell_type,
            transform=axes[row_index, 0].transAxes,
            ha="right",
            va="center",
            rotation=90,
            fontsize=cell_type_fontsize,
            fontweight="bold",
        )

    fig.tight_layout(h_pad=1.5, w_pad=0.9)
    if output_path is not None:
        save_figure(fig, output_path, dpi=dpi, bbox_inches="tight")
    return fig


def compute_cell_type_gene_summary(
    adata_obj: ad.AnnData,
    genes: list[str],
    *,
    groupby: str = ANNOTATED_CELL_TYPE_KEY,
    group_order: list[str] | None = None,
    layer: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return mean expression and fraction expressing for genes grouped by annotated cell type."""
    if groupby not in adata_obj.obs:
        raise KeyError(f"Expected adata.obs[{groupby!r}]. Run add_annotated_cell_type() first.")
    if group_order is None:
        groups = adata_obj.obs[groupby]
        if isinstance(groups.dtype, pd.CategoricalDtype):
            group_order = [str(category) for category in groups.cat.categories]
        else:
            group_order = [str(group) for group in pd.unique(groups.astype(str))]

    resolved_genes = [_resolve_gene_name(adata_obj, gene) for gene in genes]
    gene_indices = [int(adata_obj.var_names.get_loc(gene)) for gene in resolved_genes]
    matrix = adata_obj.layers[layer] if layer is not None else adata_obj.X
    expression = matrix[:, gene_indices]
    if sparse.issparse(expression):
        expression = expression.tocsr()
    else:
        expression = np.asarray(expression)

    labels = adata_obj.obs[groupby].astype(str).to_numpy()
    means: list[np.ndarray] = []
    fractions: list[np.ndarray] = []
    for group in group_order:
        mask = labels == str(group)
        if not np.any(mask):
            means.append(np.zeros(len(resolved_genes), dtype=float))
            fractions.append(np.zeros(len(resolved_genes), dtype=float))
            continue
        group_expression = expression[mask]
        if sparse.issparse(group_expression):
            means.append(np.asarray(group_expression.mean(axis=0)).ravel())
            fractions.append(np.asarray((group_expression > 0).mean(axis=0)).ravel())
        else:
            means.append(np.asarray(group_expression.mean(axis=0)).ravel())
            fractions.append(np.asarray((group_expression > 0).mean(axis=0)).ravel())

    mean_expression = pd.DataFrame(means, index=group_order, columns=resolved_genes)
    fraction_expression = pd.DataFrame(fractions, index=group_order, columns=resolved_genes)
    return mean_expression, fraction_expression


def _cluster_gene_order(
    mean_expression: pd.DataFrame,
    *,
    method: str = "average",
    metric: str = "euclidean",
) -> list[str]:
    """Cluster genes by their cell-type mean-expression profile."""
    if mean_expression.shape[1] <= 2:
        return list(mean_expression.columns)

    profiles = mean_expression.T.to_numpy(dtype=float)
    profiles = np.nan_to_num(profiles, copy=False)
    profile_means = profiles.mean(axis=1, keepdims=True)
    profile_stds = profiles.std(axis=1, keepdims=True)
    profiles = (profiles - profile_means) / np.where(profile_stds == 0, 1.0, profile_stds)
    if np.allclose(profiles, 0):
        return list(mean_expression.columns)

    gene_linkage = linkage(profiles, method=method, metric=metric, optimal_ordering=True)
    return [mean_expression.columns[index] for index in leaves_list(gene_linkage)]


def plot_shared_gene_dotplot(
    mean_expression: pd.DataFrame,
    fraction_expression: pd.DataFrame,
    output_path: Path | str | None = None,
    *,
    gene_order: list[str] | None = None,
    cluster_genes: bool = True,
    standardize_mean_by_gene: bool = True,
    z_score_clip: tuple[float, float] = (-2.0, 2.0),
    min_dot_size: float = 2.0,
    max_dot_size: float = 55.0,
    cmap: str = "RdBu_r",
    figsize: tuple[float, float] | None = None,
    x_label_fontsize: float = 10.0,
    y_label_fontsize: float = 4.5,
    title_fontsize: float = 12.0,
    dpi: int = 220,
) -> plt.Figure:
    """Plot all shared genes with annotated cell types on the x-axis."""
    if gene_order is None:
        gene_order = _cluster_gene_order(mean_expression) if cluster_genes else list(mean_expression.columns)

    group_order = list(mean_expression.index)
    mean_plot = mean_expression.loc[group_order, gene_order]
    fraction_plot = fraction_expression.loc[group_order, gene_order]

    color_values = mean_plot.copy()
    if standardize_mean_by_gene:
        gene_std = color_values.std(axis=0).replace(0, np.nan)
        color_values = color_values.sub(color_values.mean(axis=0), axis=1).div(gene_std, axis=1).fillna(0)
        color_values = color_values.clip(lower=z_score_clip[0], upper=z_score_clip[1])
        vmin, vmax = z_score_clip
        colorbar_label = "Mean expression z-score"
    else:
        vmin = 0
        vmax = float(np.nanpercentile(color_values.to_numpy(), 99.5))
        if vmax <= 0:
            vmax = 1.0
        colorbar_label = "Mean expression"

    if figsize is None:
        figsize = (max(9.0, len(group_order) * 1.05 + 2.0), max(24.0, len(gene_order) * 0.09 + 2.0))

    fig, ax = plt.subplots(figsize=figsize)
    x_positions, y_positions = np.meshgrid(np.arange(len(group_order)), np.arange(len(gene_order)))
    dot_sizes = min_dot_size + fraction_plot.T.to_numpy().ravel() * (max_dot_size - min_dot_size)
    scatter = ax.scatter(
        x_positions.ravel(),
        y_positions.ravel(),
        c=color_values.T.to_numpy().ravel(),
        s=dot_sizes,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        edgecolors="lightgray",
        linewidths=0.15,
    )

    ax.set_xticks(np.arange(len(group_order)))
    ax.set_xticklabels(group_order, rotation=35, ha="right", fontsize=x_label_fontsize)
    ax.set_yticks(np.arange(len(gene_order)))
    ax.set_yticklabels(gene_order, fontsize=y_label_fontsize)
    ax.set_xlim(-0.5, len(group_order) - 0.5)
    ax.set_ylim(len(gene_order) - 0.5, -0.5)
    ax.set_xlabel("Annotated cell type")
    ax.set_ylabel("Shared Xenium/MERSCOPE genes")
    ax.set_title("Shared gene expression by annotated cell type", fontsize=title_fontsize)
    ax.grid(axis="x", color="lightgray", linewidth=0.35, alpha=0.5)
    ax.grid(axis="y", color="lightgray", linewidth=0.25, alpha=0.35)
    for spine in ax.spines.values():
        spine.set_visible(False)

    colorbar = fig.colorbar(scatter, ax=ax, fraction=0.018, pad=0.008)
    colorbar.set_label(colorbar_label)

    legend_fractions = [0.25, 0.5, 0.75, 1.0]
    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            color="none",
            markerfacecolor="white",
            markeredgecolor="gray",
            markersize=np.sqrt(min_dot_size + fraction * (max_dot_size - min_dot_size)),
            label=f"{fraction:.0%}",
        )
        for fraction in legend_fractions
    ]
    ax.legend(
        handles=legend_handles,
        title="Fraction expressing",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        borderaxespad=0.0,
    )

    fig.tight_layout()
    if output_path is not None:
        save_figure(fig, output_path, dpi=dpi, bbox_inches="tight")
    return fig


SHARED_GENES = _load_shared_genes(SHARED_GENE_CORRESPONDENCE_PATH)
SHARED_GENES_IN_ADATA, SHARED_GENES_MISSING_FROM_ADATA = _shared_genes_in_adata(adata, SHARED_GENES)
_validate_marker_genes(MARKER_GENES, SHARED_GENES, adata)
add_annotated_cell_type(adata, CLUSTER_STYLE, cell_type_order=ANNOTATED_CELL_TYPE_ORDER)

print(f"Shared genes listed:   {len(SHARED_GENES)}")
print(f"Shared genes in adata: {len(SHARED_GENES_IN_ADATA)}")
if SHARED_GENES_MISSING_FROM_ADATA:
    print("Shared genes missing from this AnnData object:", ", ".join(SHARED_GENES_MISSING_FROM_ADATA))


In [ ]:
MARKER_UMAP_OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_marker_gene_umap_grid_interactive.png"
MARKER_UMAP_OUTPUT_PDF_PATH = MARKER_UMAP_OUTPUT_PATH.with_suffix(".pdf")

fig = plot_marker_expression_umap_grid(
    adata,
    MARKER_GENES,
    output_path=None,
    point_size=0.35,
    alpha=0.85,
    cmap="viridis",
    expression_percentile=99.5,
    figsize=(18.0, 18.0),
    gene_title_fontsize=10,
    cell_type_fontsize=12,
    axis_label_fontsize=9,
    colorbar_label_fontsize=8,
    colorbar_tick_fontsize=7,
)


In [ ]:
# Optional: save the marker-expression UMAP grid.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_marker_expression_umap_grid(
    adata,
    MARKER_GENES,
    output_path=MARKER_UMAP_OUTPUT_PATH,
    point_size=0.35,
    alpha=0.85,
    cmap="viridis",
    expression_percentile=99.5,
    figsize=(18.0, 18.0),
    gene_title_fontsize=10,
    cell_type_fontsize=12,
    axis_label_fontsize=9,
    colorbar_label_fontsize=8,
    colorbar_tick_fontsize=7,
)
print(f"Saved PNG: {MARKER_UMAP_OUTPUT_PATH}")
print(f"Saved PDF: {MARKER_UMAP_OUTPUT_PDF_PATH}")


## Shared Gene Dotplot

The dotplot below summarizes all shared genes available in the clustered AnnData object. Annotated cell types are shown on the x-axis in the requested order, and shared genes are shown on the y-axis after hierarchical clustering by their cell-type mean-expression profiles. Dot size is the fraction of cells expressing each gene in an annotated cell type; color is gene-wise z-scored mean expression across the annotated cell types.


In [ ]:
SHARED_GENE_DOTPLOT_OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_shared_gene_cell_type_dotplot.png"
SHARED_GENE_DOTPLOT_OUTPUT_PDF_PATH = SHARED_GENE_DOTPLOT_OUTPUT_PATH.with_suffix(".pdf")
SHARED_GENE_MEAN_EXPRESSION_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_shared_gene_cell_type_mean_expression.csv"
SHARED_GENE_FRACTION_EXPRESSION_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_shared_gene_cell_type_fraction_expressing.csv"

add_annotated_cell_type(adata, CLUSTER_STYLE, cell_type_order=ANNOTATED_CELL_TYPE_ORDER)
shared_gene_mean_expression, shared_gene_fraction_expression = compute_cell_type_gene_summary(
    adata,
    SHARED_GENES_IN_ADATA,
    groupby=ANNOTATED_CELL_TYPE_KEY,
    group_order=ANNOTATED_CELL_TYPE_ORDER,
)
shared_gene_clustered_order = _cluster_gene_order(shared_gene_mean_expression)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shared_gene_mean_expression.loc[:, shared_gene_clustered_order].to_csv(SHARED_GENE_MEAN_EXPRESSION_PATH)
shared_gene_fraction_expression.loc[:, shared_gene_clustered_order].to_csv(SHARED_GENE_FRACTION_EXPRESSION_PATH)

fig = plot_shared_gene_dotplot(
    shared_gene_mean_expression,
    shared_gene_fraction_expression,
    output_path=SHARED_GENE_DOTPLOT_OUTPUT_PATH,
    gene_order=shared_gene_clustered_order,
    cluster_genes=False,
    standardize_mean_by_gene=True,
    x_label_fontsize=10,
    y_label_fontsize=4.5,
    dpi=300,
)
print(f"Saved PNG: {SHARED_GENE_DOTPLOT_OUTPUT_PATH}")
print(f"Saved PDF: {SHARED_GENE_DOTPLOT_OUTPUT_PDF_PATH}")
print(f"Saved mean expression table: {SHARED_GENE_MEAN_EXPRESSION_PATH}")
print(f"Saved fraction expressing table: {SHARED_GENE_FRACTION_EXPRESSION_PATH}")
